# 🚗 Indian ANPR System - Free Google Colab Backend Server
**Hardware:** Google Cloud (12.7 GB RAM + Free GPU/CPU)  
**Status:** 100% Free, Zero Out-Of-Memory Crashes  
---
### Instructions:
1. Click the **Play (Run)** button on the code cell below.
2. Wait ~1 minute until you see **🎉 YOUR ANPR BACKEND SERVER IS VERIFIED ONLINE!**
3. Copy the generated **BACKEND URL**.
4. Open your Render website, click the **⚙️ Change** button at the bottom right, paste the URL, and click **Save & Connect**!

In [ ]:
# Step 1: Clone repository & install dependencies
REPO_URL = "https://github.com/amithneranjane/ANPR_FINAL.git"

import os, subprocess, time, re, sys, urllib.request, json

!rm -rf anpr
print("📥 Cloning latest repository from GitHub...")
!git clone $REPO_URL anpr
%cd anpr

print("📦 Installing Python dependencies (takes ~1 minute)...")
!pip install -q -r requirements.txt

# Step 2: Download Cloudflare Tunnel binary
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

# Step 3: Start Flask Server with logging
print("🚀 Launching Flask ANPR Backend Server...")
flask_log = open("flask.log", "w")
flask_proc = subprocess.Popen(["python", "app.py"], stdout=flask_log, stderr=subprocess.STDOUT)

# Step 4: Health check loop (wait until YOLO & EasyOCR are fully loaded)
print("⏳ Waiting for YOLOv8 & EasyOCR models to initialize on port 5000...")
server_ready = False
for i in range(45):
    if flask_proc.poll() is not None:
        print("❌ Flask server process terminated unexpectedly. Log output:")
        with open("flask.log", "r") as f:
            print(f.read())
        sys.exit(1)
    try:
        req = urllib.request.Request("http://127.0.0.1:5000/health")
        with urllib.request.urlopen(req, timeout=2) as resp:
            if resp.status == 200:
                server_ready = True
                break
    except Exception:
        time.sleep(1.5)
        if (i + 1) % 5 == 0:
            print(f"   ... still loading models ({i * 1.5:.0f}s elapsed)")

if not server_ready:
    print("❌ Flask server did not respond in time. Checking flask.log:")
    with open("flask.log", "r") as f:
        print(f.read())
    sys.exit(1)

print("✅ Flask Backend is READY and responding on port 5000!")

# Step 5: Start Cloudflare Tunnel and verify public HTTPS URL
print("🌐 Opening Cloudflare HTTPS Tunnel...")
cf_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:5000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

public_url = None
url_pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")

for _ in range(40):
    line = cf_proc.stdout.readline()
    if not line: continue
    m = url_pattern.search(line)
    if m:
        public_url = m.group(0)
        break
    time.sleep(0.5)

if public_url:
    # Test public URL health
    print("🔍 Testing public tunnel health...")
    for _ in range(10):
        try:
            req = urllib.request.Request(f"{public_url}/health")
            with urllib.request.urlopen(req, timeout=5) as resp:
                if resp.status == 200:
                    break
        except Exception:
            time.sleep(1)

    print("=" * 65)
    print("🎉 YOUR ANPR BACKEND SERVER IS VERIFIED ONLINE!")
    print(f"🔗 BACKEND URL: {public_url}")
    print("=" * 65)
    print("👉 HOW TO CONNECT TO YOUR RENDER WEBSITE:")
    print(f"1. Copy this link: {public_url}")
    print("2. Go to your Render website (e.g. https://your-site.onrender.com)")
    print("3. Click the '⚙️ Change' badge at the bottom-right corner.")
    print("4. Paste the link and click 'Save & Connect'. Done!")
    print("=" * 65)
else:
    print("Tunnel took too long. Starting Localtunnel fallback:")
    !npx -y localtunnel --port 5000
